# ML-08 â€” Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Praveen23-kk/FlyRank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For this ranking queue, the goal is to identify content pieces that are declining (`high_visibility_label = 1`). Since we have a yes/no question with an observed label, we will use a **Random Forest Classifier** with bounded depth (e.g., `max_depth=5`). 
A simple Random Forest fits this perfectly because:
1. It handles non-linear relationships (like optimal word count sweet spots) without needing manual feature transformations.
2. The limited depth keeps the trees readable and interpretable. 
3. We can extract feature importances easily, explaining why certain content pieces are prioritized.

In [1]:
print('Method chosen: Random Forest Classifier (max_depth=5)')

Method chosen: Random Forest Classifier (max_depth=5)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

We are predicting the `high_visibility_label` (which is 1 if `trend_direction == 'down'`, meaning the page is experiencing a traffic drop).
To create an honest split, we must use `GroupShuffleSplit` on `client_id`. This ensures that a single client's pages are strictly in either the training set or the test set. If we split randomly across all rows, the model could memorize specific client attributes or niche keywords present in both train and test sets, artificially inflating the score and causing data leakage. We also drop all "target leakage" and "future-window leakage" columns such as `trend_direction`, `trend_pct`, `is_declining_label`, `impressions_last_30d`, `impressions_prev_30d`, and any other `_last_30d` or `_prev_30d` metrics.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import json

# 1. Load Data (handle both Colab and local paths)
try:
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
except:
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 2. Create Target Label
df["high_visibility_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# 3. Define Features and Drop Leakage
leakage_cols = [
    'trend_direction', 'trend_pct', 'high_visibility_label', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]
id_cols = ['content_id', 'client_id']

# Use numeric features for simplicity and robustness
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features = [c for c in numeric_cols if c not in leakage_cols and c not in id_cols]

# 4. Handle Missing Values
df[features] = df[features].fillna(0)

# 5. Group Shuffle Split by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

X_train, y_train = train_df[features], train_df['high_visibility_label']
X_test, y_test = test_df[features], test_df['high_visibility_label']

print(f"Train size: {len(train_df)} rows, Test size: {len(test_df)} rows")
print(f"Unique clients in train: {train_df['client_id'].nunique()}")
print(f"Unique clients in test: {test_df['client_id'].nunique()}")
print(f"Features used: {len(features)}")


Train size: 19166 rows, Test size: 10834 rows
Unique clients in train: 22
Unique clients in test: 10
Features used: 23


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

We train the Random Forest on the training split with `max_depth=5` and a fixed random seed.
For a fair comparison, we evaluate both the Random Forest probabilities and the week 4 deterministic baseline on the *exact same test split*, using Precision@20, Precision@50, and Precision@100.

In [3]:
def normalize(series: pd.Series) -> pd.Series:
    vals = pd.to_numeric(series, errors="coerce").fillna(0)
    mn, mx = vals.min(), vals.max()
    if mn == mx:
        return pd.Series(np.zeros(len(vals)), index=vals.index)
    return (vals - mn) / (mx - mn)

def percentile_rank(series: pd.Series) -> pd.Series:
    vals = pd.to_numeric(series, errors="coerce").fillna(0)
    return vals.rank(method="average", pct=True).fillna(0)

# 1. Compute Baseline Score on Test Set
test_df = test_df.copy()
test_df["visibility_score"] = percentile_rank(np.log1p(test_df["impressions_90d"]))
test_df["freshness_risk_score"] = percentile_rank(test_df["days_since_last_update"])
test_df["position_opportunity_score"] = (
    (1 - normalize(test_df["avg_position"].clip(lower=1, upper=50)))
    * test_df["visibility_score"]
    * (test_df["avg_position"] > 0).astype(int)
)
test_df["depth_gap_score"] = (1 - percentile_rank(test_df["word_count"])) * test_df["visibility_score"]

test_df["baseline_score"] = (
    0.40 * test_df["visibility_score"]
    + 0.30 * test_df["freshness_risk_score"]
    + 0.25 * test_df["position_opportunity_score"]
    + 0.05 * test_df["depth_gap_score"]
).clip(0, 1)

# 2. Train Random Forest
rf = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

# 3. Predict on Test Set
test_df["rf_score"] = rf.predict_proba(X_test)[:, 1]

# 4. Evaluate Precision@K
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

base_rate = y_test.mean()

results = []
for k in [20, 50, 100]:
    baseline_p = precision_at_k(y_test, test_df["baseline_score"], k)
    rf_p = precision_at_k(y_test, test_df["rf_score"], k)
    results.append({
        "Metric": f"Precision@{k}",
        "Baseline Score": baseline_p,
        "Random Forest": rf_p,
        "Base Rate": base_rate
    })

comparison_table = pd.DataFrame(results)
print("===============================================================")
print("MODEL COMPARISON (Test Set Only)")
print("===============================================================")
print(comparison_table.to_string(index=False))


MODEL COMPARISON (Test Set Only)
       Metric  Baseline Score  Random Forest  Base Rate
 Precision@20            0.35           0.60   0.559442
 Precision@50            0.32           0.64   0.559442
Precision@100            0.35           0.68   0.559442


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Looking at the feature importances, the model relies heavily on `days_with_impressions`, `impressions_90d`, `avg_position`, and `content_age_days`. This implies that the model's core signal comes from evaluating the consistency of search presence (`days_with_impressions`) alongside raw scale (`impressions_90d`) and ranking power (`avg_position`).

When analyzing False Positives (cases where the model predicted a high probability of decline, but the content was actually stable or growing), we notice a pattern. For instance, the false positives identified have relatively recent updates (`days_since_last_update = 20`) but perhaps low consistency/scale in terms of `impressions_90d` or `avg_position`. The model may be over-penalizing content that sits outside the very top ranking positions or has moderate volume, mistakenly predicting a downward trend when the traffic is actually stabilizing or slowly growing. This happens because the model finds a broad correlation between lower engagement/impressions and decline, but misses the nuance of niche pages that naturally live at those lower absolute numbers.

In [4]:
# 1. Feature Importances
importances = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("===============================================================")
print("TOP 5 FEATURE IMPORTANCES")
print("===============================================================")
print(importances.head(5).to_string(index=False))
print("\n")

# 2. False Positives Analysis
# Sort test set by RF score descending
test_sorted = test_df.sort_values('rf_score', ascending=False)
# Get top predictions that are actually NOT declining
false_positives = test_sorted[test_sorted['high_visibility_label'] == 0].head(3)

print("===============================================================")
print("SAMPLE FALSE POSITIVES (Model predicted decline, but it was stable)")
print("===============================================================")
cols_to_show = ['content_id', 'rf_score', 'days_since_last_update', 'impressions_90d', 'avg_position', 'trend_direction']
print(false_positives[cols_to_show].to_string(index=False))


TOP 5 FEATURE IMPORTANCES
              Feature  Importance
days_with_impressions    0.224126
      impressions_90d    0.206087
         avg_position    0.133320
     content_age_days    0.105606
           word_count    0.071361


SAMPLE FALSE POSITIVES (Model predicted decline, but it was stable)
          content_id  rf_score  days_since_last_update  impressions_90d  avg_position trend_direction
content_ae3ec597b70f  0.772413                      20              285          27.2              up
content_b35afe144d55  0.771202                      20              215           8.2          stable
content_937844f485a6  0.771202                      20              112           8.5          stable


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.